Load data and libraries

In [1]:
import pandas as pd
from pathlib import Path

pm1 = pd.read_csv(
    "../data/raw/ratnapark/pm1.csv"
)

pm10 = pd.read_csv(
    "../data/raw/ratnapark/pm10.csv"
)

pm25 = pd.read_csv(
    "../data/raw/ratnapark/pm25.csv"
)

print("PM1:", pm1.shape)
print("PM10:", pm10.shape)
print("PM2.5:", pm25.shape)

PM1: (19613, 2)
PM10: (19610, 2)
PM2.5: (19627, 2)


Convert datetime

In [2]:
pm1["datetime"] = pd.to_datetime(
    pm1["datetime"],
    utc=True
)

pm10["datetime"] = pd.to_datetime(
    pm10["datetime"],
    utc=True
)

pm25["datetime"] = pd.to_datetime(
    pm25["datetime"],
    utc=True
)

merge

In [3]:
test_raw = (
    pm1
    .merge(
        pm10,
        on="datetime",
        how="outer"
    )
    .merge(
        pm25,
        on="datetime",
        how="outer"
    )
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("Shape:", test_raw.shape)

display(test_raw.head())
display(test_raw.tail())

Shape: (19642, 4)


,datetime,pm1,pm10,pm25
0,2026-08-02 00:00:00+00:00,69.500000,84.500000,81.099998
1,2026-08-02 00:01:00+00:00,120.599998,134.000000,127.500000
2,2026-08-02 00:02:00+00:00,181.300003,195.899994,190.699997
3,2026-08-02 00:03:00+00:00,218.199997,231.500000,226.800003
4,2026-08-02 00:04:00+00:00,87.199997,100.800003,89.000000


,datetime,pm1,pm10,pm25
19637,2026-08-16 05:55:00+00:00,11.1,24.400000,14.9
19638,2026-08-16 05:56:00+00:00,8.8,27.900000,12.1
19639,2026-08-16 05:57:00+00:00,8.0,24.400000,10.2
19640,2026-08-16 05:58:00+00:00,7.3,16.799999,9.4
19641,2026-08-16 05:59:00+00:00,7.5,12.300000,8.4


Daily aggregration

In [5]:
test_daily = (
    test_raw
    .set_index("datetime")
    .resample("D")
    .mean(numeric_only=True)
    .reset_index()
)

test_daily = test_daily.rename(
    columns={
        "datetime": "date",
        "pm25": "pm2_5"
    }
)

test_daily = (
    test_daily
    .sort_values("date")
    .reset_index(drop=True)
)

print("Shape:", test_daily.shape)

display(test_daily)

Shape: (15, 4)


,date,pm1,pm10,pm2_5
0,2026-08-02 00:00:00+00:00,55.195013,64.022251,58.833538
1,2026-08-03 00:00:00+00:00,56.616693,75.393556,62.634817
2,2026-08-04 00:00:00+00:00,51.808746,69.281502,58.748684
3,2026-08-05 00:00:00+00:00,35.848899,49.294396,39.631938
4,2026-08-06 00:00:00+00:00,36.590483,49.773565,41.120997
5,2026-08-07 00:00:00+00:00,17.447983,31.665229,20.730946
6,2026-08-08 00:00:00+00:00,10.267731,20.642748,13.310965
7,2026-08-09 00:00:00+00:00,8.037196,17.865232,10.405691
8,2026-08-10 00:00:00+00:00,6.991672,16.479667,9.005760
9,2026-08-11 00:00:00+00:00,11.383264,26.751389,15.107986


Create change features

In [6]:
for pollutant in ["pm2_5", "pm10"]:

    test_daily[
        f"{pollutant}_change_1"
    ] = (
        test_daily[pollutant]
        - test_daily[pollutant].shift(1)
    )

    test_daily[
        f"{pollutant}_change_3"
    ] = (
        test_daily[pollutant]
        - test_daily[pollutant].shift(3)
    )

    test_daily[
        f"{pollutant}_change_7"
    ] = (
        test_daily[pollutant]
        - test_daily[pollutant].shift(7)
    )

    test_daily[
        f"{pollutant}_pct_change_1"
    ] = (
        test_daily[pollutant]
        .pct_change(1)
    )

    test_daily[
        f"{pollutant}_pct_change_3"
    ] = (
        test_daily[pollutant]
        .pct_change(3)
    )

    test_daily[
        f"{pollutant}_pct_change_7"
    ] = (
        test_daily[pollutant]
        .pct_change(7)
    )

In [7]:
pm10_features = [
    "pm2_5_change_1",
    "pm2_5_change_3",
    "pm2_5_change_7",
    "pm2_5_pct_change_1",
    "pm2_5_pct_change_3",
    "pm2_5_pct_change_7",

    "pm10_change_1",
    "pm10_change_3",
    "pm10_change_7",
    "pm10_pct_change_1",
    "pm10_pct_change_3",
    "pm10_pct_change_7",
]

print("Number of PM10 features:", len(pm10_features))

for feature in pm10_features:
    print("-", feature)

Number of PM10 features: 12
- pm2_5_change_1
- pm2_5_change_3
- pm2_5_change_7
- pm2_5_pct_change_1
- pm2_5_pct_change_3
- pm2_5_pct_change_7
- pm10_change_1
- pm10_change_3
- pm10_change_7
- pm10_pct_change_1
- pm10_pct_change_3
- pm10_pct_change_7


Select 15 days period

In [8]:
test_period = test_daily[
    test_daily["date"].between(
        "2026-08-02",
        "2026-08-16"
    )
].copy()

test_period = (
    test_period
    .sort_values("date")
    .reset_index(drop=True)
)

print("Rows in test period:", len(test_period))

display(
    test_period[
        ["date", "pm2_5", "pm10"]
        + pm10_features
    ]
)

Rows in test period: 15


,date,pm2_5,pm10,pm2_5_change_1,pm2_5_change_3,pm2_5_change_7,pm2_5_pct_change_1,pm2_5_pct_change_3,pm2_5_pct_change_7,pm10_change_1,pm10_change_3,pm10_change_7,pm10_pct_change_1,pm10_pct_change_3,pm10_pct_change_7
0,2026-08-02 00:00:00+00:00,58.833538,64.022251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-08-03 00:00:00+00:00,62.634817,75.393556,3.801279,NaN,NaN,0.064611,NaN,NaN,11.371305,NaN,NaN,0.177615,NaN,NaN
2,2026-08-04 00:00:00+00:00,58.748684,69.281502,-3.886133,NaN,NaN,-0.062044,NaN,NaN,-6.112054,NaN,NaN,-0.081069,NaN,NaN
3,2026-08-05 00:00:00+00:00,39.631938,49.294396,-19.116746,-19.201600,NaN,-0.325399,-0.326372,NaN,-19.987106,-14.727855,NaN,-0.288491,-0.230043,NaN
4,2026-08-06 00:00:00+00:00,41.120997,49.773565,1.489059,-21.513820,NaN,0.037572,-0.343480,NaN,0.479169,-25.619991,NaN,0.009721,-0.339817,NaN
5,2026-08-07 00:00:00+00:00,20.730946,31.665229,-20.390051,-38.017738,NaN,-0.495855,-0.647125,NaN,-18.108335,-37.616272,NaN,-0.363814,-0.542948,NaN
6,2026-08-08 00:00:00+00:00,13.310965,20.642748,-7.419981,-26.320973,NaN,-0.357918,-0.664135,NaN,-11.022481,-28.651648,NaN,-0.348094,-0.581235,NaN
7,2026-08-09 00:00:00+00:00,10.405691,17.865232,-2.905274,-30.715306,-48.427848,-0.218262,-0.746949,-0.823133,-2.777516,-31.908332,-46.157019,-0.134552,-0.641070,-0.720953
8,2026-08-10 00:00:00+00:00,9.005760,16.479667,-1.399931,-11.725186,-53.629057,-0.134535,-0.565589,-0.856218,-1.385566,-15.185563,-58.913889,-0.077557,-0.479566,-0.781418
9,2026-08-11 00:00:00+00:00,15.107986,26.751389,6.102226,1.797022,-43.640698,0.677591,0.135003,-0.742837,10.271722,6.108641,-42.530113,0.623297,0.295922,-0.613874


Remove unusable rows

In [9]:
test_pm10 = test_period.dropna(
    subset=pm10_features
).copy()

test_pm10 = (
    test_pm10
    .sort_values("date")
    .reset_index(drop=True)
)

print("Valid PM10 test rows:", len(test_pm10))

display(
    test_pm10[
        ["date", "pm2_5", "pm10"]
        + pm10_features
    ]
)

Valid PM10 test rows: 8


,date,pm2_5,pm10,pm2_5_change_1,pm2_5_change_3,pm2_5_change_7,pm2_5_pct_change_1,pm2_5_pct_change_3,pm2_5_pct_change_7,pm10_change_1,pm10_change_3,pm10_change_7,pm10_pct_change_1,pm10_pct_change_3,pm10_pct_change_7
0,2026-08-09 00:00:00+00:00,10.405691,17.865232,-2.905274,-30.715306,-48.427848,-0.218262,-0.746949,-0.823133,-2.777516,-31.908332,-46.157019,-0.134552,-0.641070,-0.720953
1,2026-08-10 00:00:00+00:00,9.005760,16.479667,-1.399931,-11.725186,-53.629057,-0.134535,-0.565589,-0.856218,-1.385566,-15.185563,-58.913889,-0.077557,-0.479566,-0.781418
2,2026-08-11 00:00:00+00:00,15.107986,26.751389,6.102226,1.797022,-43.640698,0.677591,0.135003,-0.742837,10.271722,6.108641,-42.530113,0.623297,0.295922,-0.613874
3,2026-08-12 00:00:00+00:00,14.218958,26.608750,-0.889028,3.813268,-25.412980,-0.058845,0.366460,-0.641225,-0.142639,8.743518,-22.685646,-0.005332,0.489415,-0.460207
4,2026-08-13 00:00:00+00:00,14.537127,23.974670,0.318169,5.531367,-26.583870,0.022376,0.614203,-0.646479,-2.634080,7.495003,-25.798895,-0.098993,0.454803,-0.518325
5,2026-08-14 00:00:00+00:00,10.211659,17.667453,-4.325468,-4.896328,-10.519287,-0.297546,-0.324089,-0.507420,-6.307217,-9.083936,-13.997776,-0.263078,-0.339569,-0.442055
6,2026-08-15 00:00:00+00:00,10.299861,14.650798,0.088203,-3.919097,-3.011103,0.008637,-0.275625,-0.226212,-3.016655,-11.957952,-5.991950,-0.170746,-0.449399,-0.290269
7,2026-08-16 00:00:00+00:00,10.739167,18.176389,0.439305,-3.797960,0.333476,0.042652,-0.261259,0.032047,3.525591,-5.798281,0.311156,0.240642,-0.241850,0.017417


In [10]:
print("Missing values in PM10 features:")

display(
    test_pm10[pm10_features].isna().sum()
)

Missing values in PM10 features:


pm2_5_change_1        0
pm2_5_change_3        0
pm2_5_change_7        0
pm2_5_pct_change_1    0
pm2_5_pct_change_3    0
pm2_5_pct_change_7    0
pm10_change_1         0
pm10_change_3         0
pm10_change_7         0
pm10_pct_change_1     0
pm10_pct_change_3     0
pm10_pct_change_7     0
dtype: int64

Check final dates

In [11]:
print("Final PM10 test dates:")

for date in test_pm10["date"]:
    print(date.date())

Final PM10 test dates:
2026-08-09
2026-08-10
2026-08-11
2026-08-12
2026-08-13
2026-08-14
2026-08-15
2026-08-16


save dataset

In [12]:
output_path = Path("../data/processed/ratnapark/ratnapark_pm10_test_15days.csv")

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

test_pm10[
    ["date", "pm2_5", "pm10"]
    + pm10_features
].to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", test_pm10.shape)

Saved: ..\data\processed\ratnapark\ratnapark_pm10_test_15days.csv
Shape: (8, 16)
